In [47]:
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

with open('/content/trimet_stopevents_2022-12-07.html', 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f, 'html.parser')

rows = []
for tr in soup.find_all('tr'):
    cells = tr.find_all('td')
    if len(cells) != 24:
        continue

    trip_id        = cells[6].text.strip()
    vehicle_number = cells[0].text.strip()
    arrive_secs    = cells[8].text.strip()
    location_id    = cells[10].text.strip()
    ons_text       = cells[13].text.strip()
    offs_text      = cells[14].text.strip()

    if (trip_id   == 'trip_number' or
        not arrive_secs.isdigit()     or
        not ons_text.isdigit()        or
        not offs_text.isdigit()):
        continue

    arrive_secs = int(arrive_secs)
    ons         = int(ons_text)
    offs        = int(offs_text)

    base_date = datetime(2022, 12, 7)
    tstamp    = base_date + timedelta(seconds=arrive_secs)

    rows.append({
        'trip_id':        trip_id,
        'vehicle_number': vehicle_number,
        'tstamp':         tstamp,
        'location_id':    location_id,
        'ons':            ons,
        'offs':           offs,
    })

stops_df = pd.DataFrame(rows)

print("Rows loaded:", len(stops_df))

num_vehicles   = stops_df['vehicle_number'].nunique()
num_locations  = stops_df['location_id'].nunique()
min_ts, max_ts = stops_df['tstamp'].min(), stops_df['tstamp'].max()
count_boarding = (stops_df['ons'] >= 1).sum()
total_events   = len(stops_df)
pct_boarding   = (count_boarding / total_events) * 100

print(f"Distinct vehicles:             {num_vehicles}")
print(f"Unique stop locations:         {num_locations}")
print(f"Earliest timestamp:            {min_ts}")
print(f"Latest timestamp:              {max_ts}")
print(f"Events with ≥1 boarding:       {count_boarding}")
print(f"% with at least one boarding: {pct_boarding:.2f}%")


Rows loaded: 93912
Distinct vehicles:             158
Unique stop locations:         4354
Earliest timestamp:            2022-12-07 04:02:29
Latest timestamp:              2022-12-08 02:37:41
Events with ≥1 boarding:       19858
% with at least one boarding: 21.15%


In [36]:
duplicates = stops_df.duplicated()
print("Exact duplicate rows:", duplicates.sum())


Exact duplicate rows: 0


In [40]:
print("Full duplicates:", stops_df.duplicated().sum())
stops_df = stops_df.drop_duplicates()


Full duplicates: 0


In [43]:
zero_trip_ids = stops_df[stops_df['trip_id'] == '0']
print("Trip ID == 0:", len(zero_trip_ids))


Trip ID == 0: 0


In [42]:
stops_df = stops_df[stops_df['trip_id'] != '0']


In [44]:
subset_cols = ['trip_id', 'vehicle_number', 'tstamp', 'location_id']
dupes = stops_df.duplicated(subset=subset_cols)
print("Near duplicates:", dupes.sum())


Near duplicates: 0


In [45]:
dupes_by_time = stops_df.duplicated(subset=['trip_id', 'vehicle_number', 'tstamp'])
print("Tstamp-level duplicates:", dupes_by_time.sum())


Tstamp-level duplicates: 0


In [46]:
with open('/content/trimet_stopevents_2022-12-07.html', 'r') as f:
    print("Lines in file:", sum(1 for _ in f))


Lines in file: 1


In [37]:
print(stops_df.head(5))
print(stops_df.tail(5))


  trip_id vehicle_number              tstamp location_id  ons  offs
0    1175           2922 2022-12-07 16:16:54        9985    3     0
1    1330           4025 2022-12-07 14:44:09        5846    0     0
2       0           4054 2022-12-07 08:55:12        2685    2     0
3    1010           3643 2022-12-07 15:03:18        9985    0     0
4    1010           3643 2022-12-07 15:03:36        9981    2     0
      trip_id vehicle_number              tstamp location_id  ons  offs
93907    1340           4229 2022-12-08 01:09:03        2703    0     1
93908    1340           4229 2022-12-08 01:09:45        2684    0     0
93909    1340           4229 2022-12-08 01:09:59        2706    0     0
93910    1340           4229 2022-12-08 01:10:17        2707    0     0
93911    1340           4229 2022-12-08 01:10:38        2685    0     1


In [38]:
print(stops_df['ons'].describe())
print(stops_df['offs'].describe())


count   9.3912000000e+04
mean    3.8730939603e-01
std     1.0369097219e+00
min     0.0000000000e+00
25%     0.0000000000e+00
50%     0.0000000000e+00
75%     0.0000000000e+00
max     2.4000000000e+01
Name: ons, dtype: float64
count   9.3912000000e+04
mean    3.9390067297e-01
std     1.0466884499e+00
min     0.0000000000e+00
25%     0.0000000000e+00
50%     0.0000000000e+00
75%     0.0000000000e+00
max     3.3000000000e+01
Name: offs, dtype: float64


In [8]:
loc_df = stops_df[stops_df['location_id'] == loc]

num_stops_loc = len(loc_df)

num_buses_loc = loc_df['vehicle_number'].nunique()

pct_board_loc = (loc_df['ons'] >= 1).mean() * 100


veh = '4062'
veh_df = stops_df[stops_df['vehicle_number'] == veh]

num_stops_veh = len(veh_df)

total_board_veh = veh_df['ons'].sum()

total_deboard_veh = veh_df['offs'].sum()

pct_board_veh = (veh_df['ons'] >= 1).mean() * 100


print("Location 6913:")
print(f"  Stops made:                   {num_stops_loc}")
print(f"  Different buses stopping:     {num_buses_loc}")
print(f"  % with ≥1 boarding:           {pct_board_loc:.2f}%\n")

print("Vehicle 4062:")
print(f"  Stops made:                   {num_stops_veh}")
print(f"  Total passengers boarded:     {total_board_veh}")
print(f"  Total passengers deboarded:   {total_deboard_veh}")
print(f"  % of stops with ≥1 boarding:  {pct_board_veh:.2f}%")


Location 6913:
  Stops made:                   15
  Different buses stopping:     5
  % with ≥1 boarding:           13.33%

Vehicle 4062:
  Stops made:                   68
  Total passengers boarded:     26
  Total passengers deboarded:   26
  % of stops with ≥1 boarding:  16.18%


In [12]:
import pandas as pd
from scipy.stats import binomtest

p0 = (stops_df['ons'] > 0).mean()

summary = stops_df.groupby('vehicle_number').agg(
    n_events=('ons', 'size'),
    n_boardings=('ons', lambda x: (x > 0).sum())
).reset_index()

summary['boarding_rate'] = summary['n_boardings'] / summary['n_events'] * 100

summary['p'] = summary.apply(
    lambda r: binomtest(int(r['n_boardings']), int(r['n_events']), p0).pvalue,
    axis=1
)

print("count the number of stops events:\n", summary[['vehicle_number', 'n_events']], "\n")

print("count the number of stop events with at least one passenger board:\n", summary[['vehicle_number', 'n_boardings']], "\n")

print("percentage of stop events with boardings:\n", summary[['vehicle_number', 'boarding_rate']], "\n")

print("p value from binomial test:\n", summary[['vehicle_number', 'p']], "\n")


count the number of stops events:
     vehicle_number  n_events
0             2907       543
1             2909       195
2             2911       966
3             2912       462
4             2922       413
..             ...       ...
153           4236      1105
154           4238      1057
155           4239       776
156           4303       473
157           4305       397

[158 rows x 2 columns] 

count the number of stop events with at least one passenger board:
     vehicle_number  n_boardings
0             2907          103
1             2909           41
2             2911          197
3             2912          109
4             2922           84
..             ...          ...
153           4236          235
154           4238          221
155           4239          172
156           4303           91
157           4305           85

[158 rows x 2 columns] 

percentage of stop events with boardings:
     vehicle_number  boarding_rate
0             2907      18.968692
1 

In [13]:
biased = summary[summary['p'] < 0.05][['vehicle_number', 'p']]
print(biased.sort_values('p'))


    vehicle_number         p
113           3915  0.017249
70            3530  0.028077
125           3963  0.033011
103           3733  0.043074
88            3634  0.045715


In [18]:
print(stops_df.columns)


Index(['trip_id', 'vehicle_number', 'tstamp', 'location_id', 'ons', 'offs'], dtype='object')


In [ ]:
# Q: 5

In [52]:
# CSV

gps_df = pd.read_csv("/content/trimet_relpos_2022-12-07.csv")
relpos_array = gps_df['RELPOS'].dropna().values


In [3]:
import pandas as pd

gps_df = pd.read_csv('/content/trimet_relpos_2022-12-07.csv')

print(gps_df[['VEHICLE_NUMBER', 'RELPOS']])


         VEHICLE_NUMBER   RELPOS
0                  4236 -11.1830
1                  4236  -4.0465
2                  4236   1.6730
3                  4236  18.9577
4                  4236  -0.2522
...                 ...      ...
1342205            3713 -10.5823
1342206            3713  -7.8979
1342207            3713   3.3119
1342208            3713  -0.2470
1342209            3713  10.9909

[1342210 rows x 2 columns]


In [7]:
from scipy.stats import ttest_1samp

gps_df['RELPOS'] = pd.to_numeric(gps_df['RELPOS'], errors='coerce')
gps_df = gps_df.dropna(subset=['RELPOS'])

results = []
for vehicle, values in gps_df.groupby('vehicle_number')['RELPOS']:
    if len(values) > 1:
        p = ttest_1samp(values, popmean=0.0).pvalue
        results.append({'vehicle_number': vehicle, 'pvalue': p})

biased_gps = pd.DataFrame(results)
biased_gps = biased_gps[biased_gps['pvalue'] < 0.005].sort_values('pvalue')

pd.set_option('display.float_format', '{:.10e}'.format)

print(biased_gps)


   vehicle_number           pvalue
2            2911 0.0000000000e+00
8            2930 0.0000000000e+00
31           3107 0.0000000000e+00
28           3056 0.0000000000e+00
20           3022 0.0000000000e+00
..            ...              ...
7            2929 5.0057430983e-19
27           3047 1.3579949480e-17
23           3031 7.3398620925e-17
94           3644 3.0168722544e-13
65           3406 9.0608202158e-07

[158 rows x 2 columns]


In [4]:
"""
from bs4 import BeautifulSoup
import pandas as pd

with open('/content/trimet_stopevents_2022-12-07.html', 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f, 'html.parser')

rows = []
for tr in soup.find_all('tr'):
    td = tr.find_all('td')
    if len(td) != 24:
        continue

    vehicle = td[0].text.strip()
    relpos = td[23].text.strip()

    try:
        relpos = float(relpos)
    except:
        continue

    rows.append({'vehicle_number': vehicle, 'RELPOS': relpos})

gps_df = pd.DataFrame(rows)
print(gps_df.head())
"""

  vehicle_number  RELPOS
0           2922     5.0
1           4025     5.0
2           4054     2.0
3           3643     5.0
4           3643     4.0


In [27]:
"""
from scipy.stats import ttest_1samp
import pandas as pd

gps_df['RELPOS'] = pd.to_numeric(gps_df['RELPOS'], errors='coerce')
gps_df = gps_df.dropna(subset=['RELPOS'])

all_relpos = gps_df['RELPOS'].values

vehicle_groups = gps_df.groupby('vehicle_number')['RELPOS']

results = []
for vehicle, values in vehicle_groups:
    if len(values) > 1:
        p = ttest_1samp(values, popmean=0.0).pvalue
        results.append({'vehicle_number': vehicle, 'pvalue': p})

biased_gps = pd.DataFrame(results)
biased_gps = biased_gps[biased_gps['pvalue'] < 0.005].sort_values('pvalue')

pd.set_option('display.float_format', '{:.10e}'.format)
print(biased_gps)
"""


   vehicle_number           pvalue
2            2911 0.0000000000e+00
8            2930 0.0000000000e+00
31           3107 0.0000000000e+00
28           3056 0.0000000000e+00
20           3022 0.0000000000e+00
..            ...              ...
7            2929 5.0057430983e-19
27           3047 1.3579949480e-17
23           3031 7.3398620925e-17
94           3644 3.0168722544e-13
65           3406 9.0608202158e-07

[158 rows x 2 columns]


In [29]:
total_ons = stops_df['ons'].sum()
total_offs = stops_df['offs'].sum()

print("Total number of ons:", total_ons)
print("Total number of offs:", total_offs)


Total number of ons: 36373
Total number of offs: 36992


In [31]:
vehicle_totals = stops_df.groupby('vehicle_number').agg(
    vehicle_ons=('ons', 'sum'),
    vehicle_offs=('offs', 'sum')
).reset_index()

print("Per-vehicle ons and offs:")
print(vehicle_totals.head())


Per-vehicle ons and offs:
  vehicle_number  vehicle_ons  vehicle_offs
0           2907          180           182
1           2909           68            69
2           2911          357           352
3           2912          205           209
4           2922          153           150


In [33]:
from scipy.stats import chi2_contingency

results = []

for _, row in vehicle_totals.iterrows():
    observed = [
        [row['vehicle_ons'], row['vehicle_offs']],
        [total_ons - row['vehicle_ons'], total_offs - row['vehicle_offs']]
    ]
    _, pvalue, _, _ = chi2_contingency(observed)

    results.append({
        'vehicle_number': row['vehicle_number'],
        'pvalue': pvalue
    })

pvalue_df = pd.DataFrame(results)
print("Chi-Square test p-values for each vehicle:")
print(pvalue_df.head())


Chi-Square test p-values for each vehicle:
  vehicle_number           pvalue
0           2907 9.9771757700e-01
1           2909 1.0000000000e+00
2           2911 7.0638113156e-01
3           2912 1.0000000000e+00
4           2922 7.9307847397e-01


In [35]:
pvalue_df = pd.DataFrame(results)

biased_vehicles = pvalue_df[pvalue_df['pvalue'] < 0.05].sort_values('pvalue')

print(biased_vehicles.head())

   vehicle_number           pvalue
78           3576 1.8782710640e-02
28           3056 3.0134462678e-02
